# Multiclass Classification on Spiral Dataset

In this tutorial, we will use **Nanograd** to solve a non-linear 3-class classification problem on the classic **Spiral** dataset. We will:
1. Generate the spiral dataset (often used to evaluate neural network expressive power).
2. Build a Multi-Layer Perceptron (MLP) classification network.
3. Train using the **Adam** optimizer and **SoftmaxCrossEntropy** loss.
4. Visualize the 3-class decision boundaries.

---

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from nanograd import Tensor, MLP, relu
from nanograd.optim import Adam
from nanograd.loss import SoftmaxCrossEntropy

np.random.seed(42)

## 1. Generating the Spiral Dataset

We will generate a dataset consisting of three branches of spirals, representing classes 0, 1, and 2.

In [ ]:
def make_spirals(points_per_class=100, num_classes=3):
    X = np.zeros((points_per_class * num_classes, 2))
    y = np.zeros(points_per_class * num_classes, dtype='uint8')
    for j in range(num_classes):
        ix = range(points_per_class * j, points_per_class * (j + 1))
        r = np.linspace(0.0, 1, points_per_class) # radius
        t = np.linspace(j * 4, (j + 1) * 4, points_per_class) + np.random.randn(points_per_class) * 0.2 # theta
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[ix] = j
    return X, y

X_np, y_np = make_spirals(points_per_class=100, num_classes=3)

# Plot the data
plt.figure(figsize=(7, 7))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, s=40, cmap=plt.cm.Spectral, edgecolors='k')
plt.title('3-Class Spiral Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. Formatting Targets (One-Hot Encoding)

For multiclass cross entropy, we need to convert class indices to probability distributions (one-hot vectors).

In [ ]:
def to_one_hot(labels, num_classes=3):
    one_hot = np.zeros((labels.shape[0], num_classes))
    one_hot[np.arange(labels.shape[0]), labels] = 1.0
    return one_hot

y_onehot = to_one_hot(y_np)

X_tensor = Tensor(X_np)
y_tensor = Tensor(y_onehot)

## 3. Creating the MLP Classifier

We construct a network with 2 inputs, two hidden layers of 32 neurons, and 3 outputs (one for each class). We manual override the output layer activation function to identity.

In [ ]:
model = MLP([2, 32, 32, 3])

# Override last layer activation to identity (linear output logits)
model.layers[-1].activation_function = lambda x: x

# Apply Kaiming (He) normal initialization for stable training
for layer in model.layers:
    layer.weights.data = np.random.normal(0, np.sqrt(2.0 / layer.num_inputs), size=layer.weights.data.shape)
    layer.bias.data = np.zeros(layer.bias.data.shape)

optimizer = Adam(model.params(), learning_rate=0.01)
criterion = SoftmaxCrossEntropy()

## 4. Training Loop

In [ ]:
epochs = 200
loss_history = []

for epoch in range(epochs):
    # 1. Forward Pass
    logits = model(X_tensor)
    
    # 2. Loss calculation
    loss = criterion(logits, y_tensor)
    loss_history.append(loss.data.item())
    
    # 3. Reset Gradients
    optimizer.zero_grad()
    
    # 4. Backward Pass
    loss.backward()
    
    # 5. Optimizer step
    optimizer.step()
    
    # Compute accuracy
    preds = np.argmax(logits.data, axis=1)
    acc = np.mean(preds == y_np)
    
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | CrossEntropy Loss: {loss.data.item():.4f} | Train Accuracy: {acc*100:.2f}%")

# Plot loss
plt.figure(figsize=(7, 4))
plt.plot(loss_history, color='#8c564b', linewidth=2)
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 5. Visualizing the Decision Boundary

In [ ]:
# Generate grid
x_min, x_max = X_np[:, 0].min() - 0.2, X_np[:, 0].max() + 0.2
y_min, y_max = X_np[:, 1].min() - 0.2, X_np[:, 1].max() + 0.2
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_tensor = Tensor(grid_points)

# Forward pass on grid points
grid_logits = model(grid_tensor).data
Z = np.argmax(grid_logits, axis=1)
Z = Z.reshape(xx.shape)

# Plot contour fill and data points
plt.figure(figsize=(8, 8))
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.7)
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, s=40, cmap=plt.cm.Spectral, edgecolors='k')
plt.title('Spiral Dataset Decision Boundary')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, linestyle='--', alpha=0.2)
plt.show()